# DreamGrid Demo

DreamGrid encodes an RGB grid into discrete VQ codes, predicts action-conditioned futures, and uses model predictive control to choose actions.

This notebook downloads the trained models and runs:

1. A successful planning episode
2. A learned imagination visualization
3. An optional small policy benchmark

In [ ]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if not os.path.exists("/content/DreamGrid"):
    subprocess.run(
        ["git", "clone", "https://github.com/YakshithK/DreamGrid.git"],
        check=True,
    )

os.chdir("/content/DreamGrid")
print("Working directory: ", os.getcwd())

In [ ]:
%pip install -r requirements.txt

In [ ]:
from pathlib import Path
import urllib.request

checkpoint_dir = Path("checkpoints/final")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

files = {
    "vqvae.pt": (
        "https://github.com/YakshithK/DreamGrid/"
        "releases/download/v1.0.0/vqvae.pt"
    ),
    "vq_dynamics.pt": (
        "https://github.com/YakshithK/DreamGrid/"
        "releases/download/v1.0.0/vq_dynamics.pt"
    ),
}

for filename, url in files.items():
    destination = checkpoint_dir / filename
    if not destination.exists():
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, destination)

    print(filename, destination.stat().st_size, "bytes")

In [ ]:
import hashlib
import torch

expected_hashes = {
    "vqvae.pt":
        "24dd8bfb88999d79835dfbbe5d295fb4b82b9fe6fbfa902fcb9d07c86c83b4c1",
    "vq_dynamics.pt":
        "03e4a05126432b9b1e6e69928a3b829f4ee55ee23bb61fa06fadaa0322341afe",
}

for filename, expected in expected_hashes.items():
    path = checkpoint_dir / filename
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    assert actual == expected, f"Checksum failed for {filename}"
    print(filename, "verified")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: ", device)

if torch.cuda.is_available():
    print("GPU: ", torch.cuda.get_device_name(0))

In [ ]:
!python -m eval.visualizations.visualize_vq_mpc_episode \
    --seed 10001 \
    --horizon 8 \
    --candidates 1024 \
    --out_dir outputs/demo

In [ ]:
from IPython.display import Image

Image(filename(
    "outputs/demo/"
    "vq_mpc_episode_seed10001_h8_c1024.png"
))

In [ ]:
!python -m eval.visualizations.visualize_vq_imagination \
    --seed 10001 \
    --horizon 12 \
    --candidates 2048 \
    --top_k 4 \
    --out_dir outputs/demo

In [ ]:
Image(filename(
    "outputs/demo/"
    "vq_imagination_seed10001_h12_c1024_k4.png"
), width=1400)

In [ ]:
!python -m eval.planners.evaluate_vq_planners \
    --episodes 10 \
    --horizon 8 \
    --candidates 1024